In [396]:
import os, tempfile
# Avoid Mac ARM + VSCode backend/permission weirdness
os.environ.setdefault("MPLCONFIGDIR", os.path.join(tempfile.gettempdir(), "mplconfig"))

import matplotlib
matplotlib.use("Agg")  # headless, safe for bulk saving

import matplotlib.pyplot as plt
import gc, warnings
warnings.filterwarnings("ignore", category=UserWarning)

import os
import math
import numpy as np
import pandas as pd
import yfinance as yf
import mplfinance as mpf
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Optional libs
HAS_TALIB = True
try:
    import talib
except Exception:
    HAS_TALIB = False

HAS_PANDAS_TA = True
try:
    import pandas_ta as ta
except Exception:
    HAS_PANDAS_TA = False

# ==== CONFIG ====
TICKERS = [

    # --- Tech ---
    'AAPL', 'MSFT', 'NVDA', 'TSLA', 'META', 'AMZN', 'GOOGL', 'GOOG', 'AMD', 'INTC',
    'ORCL', 'CRM', 'ADBE', 'CSCO', 'SHOP', 'UBER', 'PYPL', 'NFLX', 'AVGO',
    'ZM', 'DOCU', 'SNOW', 'PLTR', 'DDOG', 'OKTA', 'CRWD', 'ZS', 'NET', 'MDB',

    # --- Semiconductors ---
    'QCOM', 'TXN', 'MU', 'TSM', 'ASML', 'AMAT', 'LRCX', 'KLAC', 'NXPI', 'ON', 'MRVL',

    # --- Financials ---
    'JPM', 'BAC', 'C', 'GS', 'MS', 'BLK', 'V', 'MA', 'AXP', 'SCHW', 'COF', 'ALLY',

    # --- Industrials & Aerospace ---
    'BA', 'CAT', 'GE', 'HON', 'DE', 'NOC', 'LMT', 'RTX', 'TDG', 'GD', 'HEI', 'TXT',

    # --- Consumer & Retail ---
    'DIS', 'HD', 'LOW', 'NKE', 'SBUX', 'MCD', 'TGT', 'COST', 'WMT', 'YUM', 'KO', 'PEP',
    'KHC', 'PG', 'CL', 'EL', 'KMB',
    

    # --- Energy ---
    'XOM', 'CVX', 'SLB', 'COP', 'OXY', 'PSX', 'EOG', 'FANG', 'MPC', 'VLO',

    # --- Pharma & Healthcare ---
    'PFE', 'MRNA', 'JNJ', 'UNH', 'LLY', 'ABBV', 'BMY', 'MRK', 'GILD', 'REGN', 'BIIB',

    # --- ETFs ---
    'SPY', 'QQQ', 'IWM', 'DIA', 'XLK', 'XLF', 'XLE', 'XLV', 'XLY', 'XLI', 'ARKK',
    'SOXX', 'SMH', 'XLRE', 'XLP', 'XLB', 'XLC',

    # --- Crypto-related equities ---
    'COIN', 'RIOT', 'MARA', 'MSTR', 'HUT', 'BTBT', 'SI',

    # --- Automakers / EV ---
    'F', 'GM', 'RIVN', 'LCID', 'NIO', 'XPEV', 'LI', 'BYDDF',

    # --- International Large Caps ---
    'BABA', 'JD', 'PDD', 'NTES', 'TCEHY', 'BIDU', 'INFY', 'MELI', 'SHOP',
     'BP', 'RIO', 'BHP', 'SHEL'
]
# ===== Shared context / scaling =====



SEGMENT_LEN = 20
WINDOW_DEDUP = 2

# ==== Bearish Stalled (very strict) ====
ATR_LEN                         = 14
SMA_LEN                         = 50
ABK_USE_TALIB= True



# Candle strength / geometry
ABK_D1_BODY_MIN_ATR           = 0.7      # day1 white body at least 0.7 ATR
ABK_D2_BODY_MIN_ATR           = 0.5      # day2 white body at least 0.5 ATR
ABK_D3_BODY_MAX_ATR           = 0.45     # day3 white body must be small (stalling)

# Body tapering (deceleration)
ABK_BODY_DEC_D2_VS_D1_MIN_FRAC = 0.18    # body2 <= body1 * (1 - 0.18)
ABK_BODY_DEC_D3_VS_D2_MIN_FRAC = 0.18    # body3 <= body2 * (1 - 0.18)

# “Open inside prior real body” (crowding)
ABK_OPEN_INSIDE_EPS_FRAC      = 0.10     # each day opens within prior real body ±10%

# Upper‑wick behavior (supply on advances)
ABK_D2_UPPER_WICK_MIN_FRAC    = 0.25     # upper2 ≥ 25% of range2
ABK_D3_UPPER_WICK_MIN_FRAC    = 0.35     # upper3 ≥ 35% of range3
ABK_D3_LOWER_WICK_MAX_FRAC    = 0.15     # lower3 ≤ 15% of range3

# Progress degradation (diminishing closes)
ABK_GAIN_DEC_MIN_FRAC         = 0.40     # (C2-C1) ≥ 0.4*ATR and (C3-C2) <= (C2-C1)*(1-0.4)

# Hygiene on day1/2 wicks (avoid spiky trend candles)
ABK_D1_WICK_MAX_ATR           = 0.7
ABK_D2_WICK_MAX_ATR           = 0.7

# Context: clear uptrend into the pattern
ABK_SLOPE_BARS                = 6
ABK_SLOPE_MIN_ATR             = 0.9      # (C - C.shift(6)) / ATR ≥ 0.9
ABK_UP_LOOKBACK               = 8
ABK_MIN_GREENS                = 5
ABK_ABOVE_SMA                 = True     # price above SMA into the pattern

# Confirmation: weakness after the block
ABK_CONFIRM_NEXT_N            = 1        # look 1 bar ahead
ABK_CONFIRM_MIN_DOWN_ATR      = 0.35     # next close ≤ C3 - 0.35*ATR

# Small noise cushion
ABK_NOISE_ATR                 = 0.05


PERIOD= 'max'
INTERVAL = '1d'

OUT_OK   = "OK"
OUT_BAD  = "BAD"
CSV_PATH = "classification.csv"
STYLE = 'yahoo'

In [397]:
def ensure_dirs():
    os.makedirs(OUT_OK, exist_ok=True)
    os.makedirs(OUT_BAD, exist_ok=True)

def download_ohlc(ticker: str) -> pd.DataFrame:
    df = yf.download(ticker, period=PERIOD, interval=INTERVAL, auto_adjust=False, progress=False)
    if df.empty:
        return df
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.droplevel(1)
    df = df[['Open', 'High', 'Low', 'Close']].copy()
    df = df.dropna()
    if hasattr(df.index, 'tz_localize') and df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    return df



import pandas as pd
import numpy as np



# --- TA‑Lib wrapper (bearish only) ---
def talib_bearish_advance_block(O, H, L, C) -> pd.Series:
    if not ABK_USE_TALIB:
        return pd.Series(False, index=C.index)
    try:
        import talib
        out = talib.CDLADVANCEBLOCK(O, H, L, C)
        return (out < 0)
    except Exception:
        return pd.Series(False, index=C.index)


# --- Fallback geometry (strict) ---
def fallback_bearish_advance_block(O, H, L, C, atr: pd.Series) -> pd.Series:
    # aliases (i-2, i-1, i)
    O1,H1,L1,C1 = O.shift(2), H.shift(2), L.shift(2), C.shift(2)
    O2,H2,L2,C2 = O.shift(1), H.shift(1), L.shift(1), C.shift(1)
    O3,H3,L3,C3 = O,        H,        L,        C

    # colors
    w1, w2, w3 = C1 > O1, C2 > O2, C3 > O3

    # bodies/ranges
    body1 = (C1 - O1).clip(lower=0)
    body2 = (C2 - O2).clip(lower=0)
    body3 = (C3 - O3).clip(lower=0)

    rng1  = (H1 - L1).clip(lower=1e-9)
    rng2  = (H2 - L2).clip(lower=1e-9)
    rng3  = (H3 - L3).clip(lower=1e-9)

    # wicks
    upper1 = H1 - pd.concat([O1, C1], axis=1).max(axis=1)
    lower1 = pd.concat([O1, C1], axis=1).min(axis=1) - L1
    upper2 = H2 - pd.concat([O2, C2], axis=1).max(axis=1)
    lower2 = pd.concat([O2, C2], axis=1).min(axis=1) - L2
    upper3 = H3 - pd.concat([O3, C3], axis=1).max(axis=1)
    lower3 = pd.concat([O3, C3], axis=1).min(axis=1) - L3

    # strength & taper
    d1_ok = w1 & (body1 >= ABK_D1_BODY_MIN_ATR * atr)
    d2_ok = w2 & (body2 >= ABK_D2_BODY_MIN_ATR * atr)
    d3_small = w3 & (body3 <= ABK_D3_BODY_MAX_ATR * atr)

    taper_ok = (body2 <= body1 * (1 - ABK_BODY_DEC_D2_VS_D1_MIN_FRAC)) & \
               (body3 <= body2 * (1 - ABK_BODY_DEC_D3_VS_D2_MIN_FRAC))

    # opens crowd inside prior real body
    prev_low_body_2  = pd.concat([O1, C1], axis=1).min(axis=1)
    prev_high_body_2 = pd.concat([O1, C1], axis=1).max(axis=1)
    open2_inside = (O2 >= prev_low_body_2 - ABK_OPEN_INSIDE_EPS_FRAC*rng1) & \
                   (O2 <= prev_high_body_2 + ABK_OPEN_INSIDE_EPS_FRAC*rng1)

    prev_low_body_3  = pd.concat([O2, C2], axis=1).min(axis=1)
    prev_high_body_3 = pd.concat([O2, C2], axis=1).max(axis=1)
    open3_inside = (O3 >= prev_low_body_3 - ABK_OPEN_INSIDE_EPS_FRAC*rng2) & \
                   (O3 <= prev_high_body_3 + ABK_OPEN_INSIDE_EPS_FRAC*rng2)

    # increasing upper wicks, tiny lower on day3 (supply)
    wicks_ok = (upper2 >= ABK_D2_UPPER_WICK_MIN_FRAC * rng2) & \
               (upper3 >= ABK_D3_UPPER_WICK_MIN_FRAC * rng3) & \
               (lower3 <= ABK_D3_LOWER_WICK_MAX_FRAC * rng3)

    # day1/2 wick hygiene
    wick12_ok = (upper1 <= ABK_D1_WICK_MAX_ATR * atr) & (lower1 <= ABK_D1_WICK_MAX_ATR * atr) & \
                (upper2 <= ABK_D2_WICK_MAX_ATR * atr) & (lower2 <= ABK_D2_WICK_MAX_ATR * atr)

    # progress degradation
    gain12 = (C2 - C1)
    gain23 = (C3 - C2)
    prog_ok = (gain12 >= ABK_GAIN_DEC_MIN_FRAC*atr) & \
              (gain23 <= gain12 * (1 - ABK_GAIN_DEC_MIN_FRAC) + ABK_NOISE_ATR*atr)

    geom = d1_ok & d2_ok & d3_small & taper_ok & open2_inside & open3_inside & wicks_ok & wick12_ok & prog_ok
    return geom.fillna(False)



def atr_from_ohlc(H: pd.Series, L: pd.Series, C: pd.Series, n: int = 14) -> pd.Series:
    """
    Compute Average True Range (ATR) from OHLC series.
    Uses Wilder's smoothing (EMA-like) over `n` periods.
    """
    # Previous close
    prev_close = C.shift(1)

    # True Range (TR)
    tr1 = H - L
    tr2 = (H - prev_close).abs()
    tr3 = (L - prev_close).abs()
    tr  = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    # ATR: Wilder's moving average
    atr = tr.ewm(span=n, min_periods=n, adjust=False).mean()
    return atr

# --- Context: firm uptrend into day3 ---
def context_filter(df: pd.DataFrame) -> pd.Series:
    O,H,L,C = df['Open'], df['High'], df['Low'], df['Close']
    atr = atr_from_ohlc(H, L, C, n=ATR_LEN)

    slope_ok = ((C - C.shift(ABK_SLOPE_BARS)) / atr) >= ABK_SLOPE_MIN_ATR
    greens   = (C > O).rolling(ABK_UP_LOOKBACK).sum() >= ABK_MIN_GREENS

    if ABK_ABOVE_SMA:
        sma = C.rolling(SMA_LEN, min_periods=SMA_LEN//2).mean()
        above = C > sma
    else:
        above = pd.Series(True, index=C.index)

    # evaluate context up to i-1; align to i
    return (slope_ok & greens & above).shift(1).fillna(False)


# --- Confirmation: follow‑through down ---
def advance_block_confirm(C: pd.Series, atr: pd.Series) -> pd.Series:
    if ABK_CONFIRM_NEXT_N and ABK_CONFIRM_NEXT_N > 0:
        nextC = C.shift(-ABK_CONFIRM_NEXT_N)
        return (nextC <= (C - ABK_CONFIRM_MIN_DOWN_ATR*atr)).fillna(False)
    return pd.Series(True, index=C.index)


# --- Final conservative signal ---
def base_signal(df: pd.DataFrame) -> pd.Series:
    O,H,L,C = df['Open'], df['High'], df['Low'], df['Close']
    atr = atr_from_ohlc(H, L, C, n=ATR_LEN)

    tal = talib_bearish_advance_block(O, H, L, C).astype(bool)
    fb  = fallback_bearish_advance_block(O, H, L, C, atr).astype(bool)
    ctx = context_filter(df).astype(bool)
    cnf = advance_block_confirm(C, atr).astype(bool)

    return (tal & fb & ctx & cnf).astype(bool)









def dedupe_indices(indices, window_dedup):
    """
    Given a sorted list of candidate indices, remove near-duplicates
    closer than `window_dedup` bars apart.
    """
    if not indices:
        return []
    deduped = [indices[0]]
    for idx in indices[1:]:
        if idx - deduped[-1] >= window_dedup:
            deduped.append(idx)
    return deduped

def plot_segment(df: pd.DataFrame, center_loc: int, out_path: str) -> None:
    half  = SEGMENT_LEN // 2
    start = max(0, center_loc - half)
    end   = min(len(df), center_loc + half + 1)
    seg   = df.iloc[start:end][['Open','High','Low','Close']]
    if len(seg) < 3:
        return
    fig, _ = mpf.plot(
        seg,
        type='candle',
        style=STYLE,
        returnfig=True,
        tight_layout=True
    )
    fig.savefig(out_path, dpi=140, bbox_inches='tight', pad_inches=0.1)
    plt.close(fig)

In [398]:
ensure_dirs()
rows = []

good_counter = 0
bad_counter  = 0

for ticker in TICKERS:
    df = download_ohlc(ticker)
    if df.empty or len(df) < max(SMA_LEN, ATR_LEN) + SEGMENT_LEN + 2:
        print(f"{ticker}: skipped (not enough data)")
        continue

    O, H, L, C = df['Open'], df['High'], df['Low'], df['Close']

    #COMPUTING ATR
    O, H, L, C = df['Open'], df['High'], df['Low'], df['Close']

    # define atr here (same as in context_filters)
    tr  = pd.concat([(H - L), (H - C.shift()).abs(), (L - C.shift()).abs()], axis=1).max(axis=1)
    atr = tr.ewm(alpha=1/ATR_LEN, adjust=False).mean().clip(lower=1e-9)

    cand = (
    talib_bearish_advance_block(O, H, L, C).astype(bool)
    | fallback_bearish_advance_block(O, H, L, C, atr).astype(bool)
    )


    # --- Strict Morning Doji Star context (your context_filters must check MDS rules) ---
    ctx_ok = context_filter(df).astype(bool)

    ok_mask  = cand & ctx_ok            # keep
    bad_mask = cand & (~ctx_ok)         # reject, but still save to NOT_ folder

    print(f"{ticker}: cand={int(cand.sum())} ok={int(ok_mask.sum())} bad={int(bad_mask.sum())}")

    # dedupe across ALL candidates (so we save both good and bad)
    all_idxs  = np.where((ok_mask | bad_mask).values)[0].tolist()
    cand_idxs = dedupe_indices(all_idxs, WINDOW_DEDUP)

    half = SEGMENT_LEN // 2

    for i in cand_idxs:
        # stop at 200 accepted (good) images total
        if good_counter >= 200:
            break

        # skip edges that can’t render a full segment
        if i < half or i > len(df) - half - 1:
            rows.append([f"SKIP_{ticker}_{df.index[i]}", ticker, df.index[i], False, "edge_skip"])
            continue

        accepted = bool(ok_mask.iloc[i])
        folder   = OUT_OK if accepted else OUT_BAD

        if accepted:
            good_counter+= 1
            fname = f"BML_{good_counter}.png"
        else:
            bad_counter  += 1
            fname = f"NOT_{bad_counter}.png"

        out_path = os.path.join(folder, fname)

        try:
            plot_segment(df, i, out_path)
            if not os.path.exists(out_path):
                rows.append([fname, ticker, df.index[i], accepted, "plot_skipped"])
                continue
        except Exception as e:
            rows.append([fname, ticker, df.index[i], accepted, f"plot_error: {e}"])
            continue

        rows.append([fname, ticker, df.index[i], accepted, "ok" if accepted else "failed_context"])

# Save CSV log
csv_df = pd.DataFrame(rows, columns=["filename", "ticker", "timestamp", "accepted", "reason"])
csv_df.sort_values(["ticker", "timestamp"], inplace=True)
csv_df.to_csv(CSV_PATH, index=False)

print("Done.")
print(f"Accepted images saved: {good_counter}")
print(f"Rejected images saved: {bad_counter}")
print(f"- Saved to: {OUT_OK}/ and {OUT_BAD}/")
print(f"- Log: {CSV_PATH}")

AAPL: cand=57 ok=29 bad=28
MSFT: cand=61 ok=27 bad=34
NVDA: cand=32 ok=16 bad=16
TSLA: cand=11 ok=3 bad=8
META: cand=20 ok=7 bad=13
AMZN: cand=45 ok=24 bad=21
GOOGL: cand=25 ok=11 bad=14
GOOG: cand=26 ok=11 bad=15
AMD: cand=51 ok=25 bad=26
INTC: cand=81 ok=43 bad=38
ORCL: cand=57 ok=29 bad=28
CRM: cand=44 ok=16 bad=28
ADBE: cand=66 ok=25 bad=41
CSCO: cand=47 ok=22 bad=25
SHOP: cand=18 ok=9 bad=9
UBER: cand=5 ok=2 bad=3
PYPL: cand=13 ok=5 bad=8
NFLX: cand=40 ok=19 bad=21
AVGO: cand=17 ok=7 bad=10
ZM: cand=6 ok=1 bad=5
DOCU: cand=8 ok=1 bad=7
SNOW: cand=7 ok=3 bad=4
PLTR: cand=8 ok=5 bad=3
DDOG: cand=10 ok=5 bad=5
OKTA: cand=20 ok=9 bad=11
CRWD: cand=8 ok=3 bad=5
ZS: cand=15 ok=8 bad=7
NET: cand=15 ok=8 bad=7
MDB: cand=9 ok=2 bad=7
QCOM: cand=26 ok=9 bad=17
TXN: cand=76 ok=42 bad=34
MU: cand=65 ok=27 bad=38
TSM: cand=29 ok=9 bad=20
ASML: cand=17 ok=9 bad=8
AMAT: cand=48 ok=23 bad=25
LRCX: cand=66 ok=30 bad=36
KLAC: cand=62 ok=36 bad=26
NXPI: cand=9 ok=4 bad=5
ON: cand=35 ok=11 bad=24
MRV

KeyboardInterrupt: 

In [ ]:
print("OUT_OK =", repr(OUT_OK))
print("OUT_BAD =", repr(OUT_BAD))
assert OUT_OK != OUT_BAD, "OUT_OK and OUT_BAD must be different!"

OUT_OK = 'OK'
OUT_BAD = 'BAD'
